<div style="background: linear-gradient(135deg, #050010 0%, #0a0025 35%, #002020 70%, #001510 100%);padding: 60px 40px; border-radius: 18px; text-align: center; margin-bottom: 10px; border: 1px solid #00ffaa33;"><h1 style="color: #00ffaa; font-family: 'Courier New', monospace; font-size: 2.9em;letter-spacing: 5px; margin: 0 0 12px 0; text-shadow: 0 0 40px #00ffaa55;">🎭 OpenCV DEVAM</h1><h2 style="color: #a8dadc; font-family: 'Courier New', monospace;font-size: 1.35em; margin: 0 0 18px 0; font-weight: 300;">7. Hafta — Renk Uzayları, HSV Maskeleme, Perspektif & Eşikleme</h2><p style="color: #6b7280; font-family: 'Courier New', monospace; font-size: 0.85em; margin: 0;">Renk Kanalları &bull; HSV Takip &bull; Perspektif Dönüşüm &bull; Eşikleme &bull; Adaptif Eşikleme &bull; Şeffaflık</p></div>

---
# 🎨 BÖLÜM 1: Renk Uzaylarının Anatomisi

## 1.1 Neden Birden Fazla Renk Uzayı Var?

Her renk uzayı farklı bir amaca hizmet eder:

```
BGR  → OpenCV'nin yerli dili. Donanım kökenli.
RGB  → İnsan sezgisi. Ekran, web standardı.
HSV  → Renk takibi. Aydınlatmadan bağımsız.
LAB  → İnsan algısı. Renk farkı ölçümü.
YCrCb→ Video sıkıştırma. JPEG, H.264.
CMYK → Baskı. Mürekkep bazlı.
XYZ  → CIE standart. Tüm uzayların referansı.
```

**Temel soru:** Görevinize göre doğru uzayı seçmek hem doğruluk hem performans açısından kritiktir.

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Renk Biliminin Tarihi — Newton'dan Dijital Çağa:</b><br><br>1666'da Isaac Newton, güneş ışığını prizmadan geçirerek gökkuşağı elde etti ve rengin ışığın doğasında olduğunu kanıtladı. Rengin matematiksel tanımlanması için 200 yıl daha gerekti.<br><br>1931'de CIE (Uluslararası Aydınlatma Komisyonu) insan görsel sistemini deneylerle ölçerek <b>CIE XYZ renk uzayını</b> tanımladı. Bu hâlâ tüm dijital renk uzaylarının matematiksel temelidir.<br><br><b>HSV</b>'yi 1978'de Alvy Ray Smith geliştirdi — daha sonra Pixar'ın kurucu ortağı oldu. HSV, sanatçıların "renk seç, koyuluğunu ayarla" sezgisini matematikleştirdi. Adobe Photoshop'taki renk seçici hâlâ HSV/HSL tabanlıdır.<br><br><b>YCrCb</b>: 1938'de siyah-beyaz TV ile renkli TV'nin geriye dönük uyumlu olması için geliştirildi. Y=parlaklık, eski TV'ler yalnızca Y'yi görür. Bugün hâlâ JPEG ve H.264'ün içinde yaşıyor.</span></div>

## 1.2 Renk Kanallarını Ayrı Ayrı İnceleme

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

resim = cv.imread("resim/kirpik_bugday.jpg")
print("Görüntü boyutu:", resim.shape)

# BGR kanallarına ayrıştır
b_kanal = resim[:, :, 0]   # Mavi
g_kanal = resim[:, :, 1]   # Yeşil
r_kanal = resim[:, :, 2]   # Kırmızı

# Kanal istatistikleri
for isim, kanal in [("Mavi", b_kanal), ("Yeşil", g_kanal), ("Kırmızı", r_kanal)]:
    print(f"{isim:8}: ort={kanal.mean():.1f}, std={kanal.std():.1f}, "
          f"min={kanal.min()}, max={kanal.max()}")

# OpenCV penceresi: her kanalı ayrı göster
cv.imshow("mavi",    b_kanal)
cv.imshow("yesil",   g_kanal)
cv.imshow("kirmizi", r_kanal)
cv.waitKey(0)
cv.destroyAllWindows()

# Matplotlib ile histogramlar
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
renkler_bgr = [("Mavi", b_kanal, "blue"),
               ("Yeşil", g_kanal, "green"),
               ("Kırmızı", r_kanal, "red")]
for ax, (isim, kanal, renk) in zip(axes, renkler_bgr):
    ax.hist(kanal.ravel(), bins=256, range=(0,256), color=renk, alpha=0.75)
    ax.set_title(f"{isim} Kanalı Histogramı")
    ax.set_xlabel("Piksel Değeri")
    ax.set_ylabel("Piksel Sayısı")
plt.suptitle("BGR Kanal Histogramları", fontsize=12)
plt.tight_layout()
plt.show()

---
# 🔄 BÖLÜM 2: Renk Uzayı Dönüşümleri

## 2.1 BGRA — Alpha Kanallı Görüntüler

4 kanallı TIF/PNG görüntülerle çalışma:

In [ ]:
import cv2 as cv

# IMREAD_UNCHANGED: alpha kanalı dahil tüm kanalları oku
resim = cv.imread("resim/b.tif", cv.IMREAD_UNCHANGED)
print("BGRA shape:", resim.shape)   # (H, W, 4)

# BGRA → Gri
gri = cv.cvtColor(resim, cv.COLOR_BGRA2GRAY)
print("Gri shape:", gri.shape)      # (H, W)

cv.imshow("a", gri)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
import cv2 as cv

# BGR → BGRA: alpha kanalı ekle
resim = cv.imread("resim/a.png", cv.IMREAD_UNCHANGED)
print("Orijinal:", resim.shape)

seffaf = cv.cvtColor(resim, cv.COLOR_BGR2BGRA)
print("BGRA:", seffaf.shape)

# Alpha kanalı başlangıçta 255 (tamamen opak)
print("Alpha örnek:", seffaf[0, :5, 3])   # ilk satırın ilk 5 pikseli

cv.imshow("a", seffaf)
cv.waitKey(0)
cv.imwrite("resim/a_alpha_kanalli.png", seffaf)
cv.destroyAllWindows()

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Alpha Kanalı ve Premultiplied Alpha:</b><br><br>Standart (straight) alpha: piksel değerleri alpha'dan bağımsız saklanır. Formül: <code>sonuç = src × alpha + dst × (1 - alpha)</code><br><br><b>Premultiplied alpha:</b> Piksel değerleri zaten alpha ile çarpılmış saklanır. Birleştirme: <code>sonuç = src_premul + dst × (1 - alpha)</code><br>Avantaj: kenar yumuşatma (anti-aliasing) daha doğal görünür, render pipeline'da çarpma işlemi azalır.<br><br>OpenCV straight alpha kullanır. Unreal Engine, Core Animation (iOS) ve çoğu GPU pipeline premultiplied kullanır. İkisi arasındaki dönüşüm yapılmadan birleştirme "halo" efekti yaratır — web'de sıkça karşılaşılan transparan PNG kenarlardaki beyaz çizgi sorunu budur!</span></div>

---
# 🫥 BÖLÜM 3: Görüntü Arka Planını Şeffaf Yapma

## 3.1 Parlaklık Bazlı Şeffaflık Oluşturma

Belirli parlaklık eşiğinin üzerindeki pikselleri şeffaf yapma:

In [ ]:
import cv2 as cv
import numpy as np

resim     = cv.imread("resim/seffaf_olacak.jpg", cv.IMREAD_UNCHANGED)
resim_gri = cv.imread("resim/seffaf_olacak.jpg", cv.IMREAD_GRAYSCALE)

# BGR → BGRA: şeffaflık kanalı ekle
seffaf = cv.cvtColor(resim, cv.COLOR_BGR2BGRA)

print("BGRA görüntü boyutu:", seffaf.shape)
print("Alpha kanalı başlangıç değerleri (ilk satır, ilk 5 piksel):")
print(seffaf[0, :5, 3])   # hepsi 255 (opak)

# Piksel piksel: parlaklık ≥ 150 ise şeffaf yap (alpha=0)
for x in range(0, 480):
    for y in range(0, 640):
        if resim_gri[x, y] >= 150:
            seffaf[x, y, 3] = 0   # alpha=0: tamamen şeffaf

print("İşlem sonrası alpha değerleri:")
print(seffaf[0, :5, 3])   # bazıları 0 olacak

cv.imshow("a", seffaf)
cv.waitKey(0)
cv.imwrite("resim/a_alpha_kanalli.png", seffaf)
cv.destroyAllWindows()

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Piksel Döngüsü → NumPy Vektörizasyonu:</b><br><br>Yukarıdaki iç içe for döngüsü 480×640 = 307,200 iterasyon yapar. Python'da yavaştır. Aynı işlemi NumPy ile tek satırda yapabilirsiniz:<br><br><code># Maske oluştur: parlaklık ≥ 150 olan piksel konumları<br>maske = resim_gri >= 150<br># O konumlardaki alpha değerini 0 yap<br>seffaf[maske, 3] = 0</code><br><br>Hız farkı: Python döngüsü ~2-5 saniye, NumPy boolean indexing ~2 milisaniye. <b>1000x+ hız artışı</b> — görüntü işlemede vektörizasyon altın kuraldır!</span></div>

---
# 🔵 BÖLÜM 4: HSV ile Renk Takibi

## 4.1 HSV Renk Uzayı — Derinlemesine

```
HSV Silindiri:
        Beyaz
          │ V=1
          │
   S=0 ───┼─── S=1   → Doygunluk (Saturation)
          │
          │ V=0
        Siyah

H (Hue): Renk tonu — 0°-360° renk çemberi
OpenCV'de H: 0-179 (360° / 2 = daraltılmış)
S: 0-255 (0=renksiz/gri, 255=canlı renk)
V: 0-255 (0=siyah, 255=en parlak)

HSV Renk Aralıkları (OpenCV):
Kırmızı  : [0-10]  ve  [160-179]
Turuncu  : [10-25]
Sarı     : [25-35]
Yeşil    : [35-85]
Cyan     : [85-100]
Mavi     : [100-130]
Mor      : [130-160]
```

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 Neden Kırmızı İki Aralıkta?</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">HSV renk çemberinde Hue 0°'den 360°'ye döner. Kırmızı renk çemberin hem başında (0°) hem sonunda (360°=0°) yer alır. OpenCV'de H değeri 0-179 arasındadır (gerçek H/2). Bu yüzden kırmızı hem 0-10 hem de 160-179 aralığını kapsar.<br><br>Çözüm: İki ayrı maske oluşturup birleştirin:<br><code>mask1 = cv.inRange(hsv, (0,100,100), (10,255,255))<br>mask2 = cv.inRange(hsv, (160,100,100), (179,255,255))<br>mask = cv.bitwise_or(mask1, mask2)</code><br><br>Bu detayı atlamak, kırmızı takip uygulamalarında rengin yarısının kaçırılmasına yol açar!</span></div>

## 4.2 Mavi Nesne Takibi — Kameradan

Kamera akışında mavi renkli nesneleri tespit edip izole etme:

In [ ]:
import numpy as np
import cv2

capture = cv2.VideoCapture(0)

# HSV'de mavi renk aralığı
lower_blue = np.array([100, 110, 110])
upper_blue = np.array([130, 255, 255])

print("Mavi nesne takibi başlıyor — q ile çık")
print(f"HSV aralığı: {lower_blue} → {upper_blue}")

while True:
    ret, frame = capture.read()
    if not ret:
        break

    # BGR → HSV dönüşümü
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Renk maskesi: aralıktaki piksel=255, dışı=0
    mask = cv2.inRange(hsv, lower_blue, upper_blue)

    # Maskeyi görüntüye uygula: sadece mavi pikseller kalsın
    res = cv2.bitwise_and(frame, frame, mask=mask)

    cv2.imshow("frame",  frame)   # orijinal
    cv2.imshow("mask",   mask)    # ikili maske (siyah-beyaz)
    cv2.imshow("res",    res)     # maskelenmiş renkli

    if cv2.waitKey(1) == ord("q"):
        break

capture.release()
cv2.destroyAllWindows()

## 4.3 Kırmızı Nesne Takibi — inRange ile Maskeleme

Kameradan kırmızıyı beyaza, diğerlerini siyaha dönüştürme:

In [ ]:
import cv2 as cv
import numpy as np

video = cv.VideoCapture(0)

# Kırmızı: iki aralık (renk çemberi başı ve sonu)
lower_red = np.array([160, 100, 100])
upper_red = np.array([179, 255, 255])

print("Kırmızı takip: kırmızı=beyaz, diğerleri=siyah — q ile çık")

while video.isOpened():
    ret, frame = video.read()
    if not ret:
        break

    hsv  = cv.cvtColor(frame, cv.COLOR_BGR2HSV)
    mask = cv.inRange(hsv, lower_red, upper_red)
    # mask: kırmızı pikseller=255 (beyaz), diğerleri=0 (siyah)

    cv.imshow("kamera", mask)
    if cv.waitKey(33) == ord("q"):
        break

video.release()
cv.destroyAllWindows()

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">🔄 Karşılaştırma</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>BGR ile renk tespiti vs HSV ile renk tespiti:</b><br><br><b>BGR yaklaşımı:</b> Kırmızı için R>150, G<50, B<50 şartı.
Sorun: Parlaklık değişirse (gölge, farklı ışık) R değeri düşer, G ve B yükselir → tespit başarısız.<br><br><b>HSV yaklaşımı:</b> H (renk tonu) aydınlatmadan büyük ölçüde bağımsızdır. Güneş ışığında ve yapay ışıkta kırmızı H≈0 veya H≈175 olmaya devam eder. S (doygunluk) sayesinde gri/beyaz tonlar elenir. V (parlaklık) alt sınırı ile siyah elenir.<br><br>Sonuç: Endüstriyel robotik, konveyör bant renk sınıflandırma, trafik işareti tespiti — hepsinde HSV tercih edilir.</span></div>

---
# 📐 BÖLÜM 5: Perspektif Dönüşümü — `warpPerspective`

## 5.1 Neden Perspektif Düzeltme?

Kamera eğik tutulduğunda veya yüzey düz olmadığında görüntü çarpıklaşır. Perspektif dönüşüm 4 noktayı istenen konumlara eşleyerek düzeltilebilir.

```
Eğik Görüntü:           Perspektif Düzeltme Sonrası:

 A──────B                A────────────B
  \    /     →           |            |
   \  /                  |            |
    C──D                 C────────────D

4 kaynak nokta (A,B,C,D) → 4 hedef nokta
getPerspectiveTransform() → 3×3 H matrisi
warpPerspective() → dönüşüm uygula
```

In [ ]:
import cv2 as cv
import numpy as np
import tqdm

fourcc = cv.VideoWriter_fourcc(*"XVID")
out = cv.VideoWriter("video/fare_analiz.avi", fourcc, 30.0, (500, 500))

video = cv.VideoCapture("video/F7of.mov")
i = 0
yesil_yol = []

total_frame = int(video.get(cv.CAP_PROP_FRAME_COUNT))
frame_no = 0

while video.isOpened():
    ret, frame = video.read()
    if not ret:
        break
    i += 1
    if i < 270:          # ilk 270 kare: kurulum öncesi, atla
        continue

    # ── Gri tonlama + ROI kırpma ──────────────────────────────────────
    grayFrame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    grayFrame = grayFrame[75:910, 430:1320]

    # ── Perspektif düzeltme ───────────────────────────────────────────
    # Kaynak: kırpılmış görüntüdeki 4 köşe (elle belirlendi)
    pts1 = np.float32([[74, 5], [860, 53], [40, 786], [806, 825]])
    # Hedef: 500×500 mükemmel dikdörtgen
    pts2 = np.float32([[0, 0], [500, 0], [0, 500], [500, 500]])

    M = cv.getPerspectiveTransform(pts1, pts2)
    grayFrame = cv.warpPerspective(grayFrame, M, (500, 500))

    # ── Gürültü azaltma ───────────────────────────────────────────────
    grayFrame = cv.medianBlur(grayFrame, 5)
    grayFrame = cv.medianBlur(grayFrame, 5)

    # ── Eşikleme: fareyi bul ──────────────────────────────────────────
    ret2, esiklenmis = cv.threshold(grayFrame, 180, 255, cv.THRESH_BINARY)

    # ── Farenin bounding box'ı ────────────────────────────────────────
    x1, y1, x2, y2 = 1000, 1000, 0, 0
    for row in range(esiklenmis.shape[0]):
        for col in range(esiklenmis.shape[1]):
            if esiklenmis[row][col] > 180:
                x1 = min(x1, col)
                y1 = min(y1, row)
                x2 = max(x2, col)
                y2 = max(y2, row)
    y = round((y1 + y2) / 2)
    x = round((x1 + x2) / 2)
    yesil_yol.append((x, y))

    # ── Yol çizimi ───────────────────────────────────────────────────
    renkli = cv.cvtColor(esiklenmis, cv.COLOR_GRAY2BGR)
    for s, yol in enumerate(yesil_yol):
        if s > 0:
            cv.line(renkli, yesil_yol[s-1], yesil_yol[s], (0, 255, 0), 2)
    renkli[y-3:y+3, x-3:x+3] = [0, 0, 255]   # merkez nokta

    out.write(renkli)
    frame_no += 1

    if cv.waitKey(33) == 27:
        break

out.release()
video.release()
cv.destroyAllWindows()
print(f"Analiz tamamlandı: {frame_no} kare işlendi")

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Medyan Bulanıklaştırma vs Gaussian Bulanıklaştırma:</b><br><br><b>Gaussian Blur:</b> Her piksel komşularının ağırlıklı ortalamasıdır. Gürültüyü azaltır ama "tuz-biber" (salt-and-pepper) gürültüye karşı zayıftır çünkü uç değerler (0 veya 255) ortalamayı bozar.<br><br><b>Median Blur:</b> Her piksel komşularının medyanı (ortanca değeri) alınır. "Tuz-biber" gürültüsüne karşı çok daha güçlüdür. Neden? Medyan, tek bir uç değerden etkilenmez.<br><br>Fare takip örneğinde iki kez medyan blur uygulanması arena aydınlatmasından kaynaklanan "ışık lekelerini" etkili biçimde temizler. Tıbbi görüntülemede (MRI, ultrason) medyan filtre standart ön işleme adımıdır.</span></div>

---
# 🔲 BÖLÜM 6: Eşikleme (Thresholding)

## 6.1 Eşikleme Nedir?

Gri tonlamalı görüntüyü ikili (0/255) görüntüye dönüştürme işlemidir. Piksel değerleri belirli bir eşiğe (threshold) göre siyah veya beyaza çevrilir.

```python
ret, dst = cv.threshold(src, thresh, maxval, type)
                         ↑       ↑         ↑
                     eşik değeri  max değer  eşikleme tipi
```

| Tip | Formül | Kullanım |
|-----|--------|----------|
| `THRESH_BINARY` | dst = maxval if src>thresh else 0 | Belge tarama |
| `THRESH_BINARY_INV` | dst = 0 if src>thresh else maxval | Ters ikili |
| `THRESH_TRUNC` | dst = thresh if src>thresh else src | Kırpma |
| `THRESH_TOZERO` | dst = src if src>thresh else 0 | Sıfırlama |
| `THRESH_TOZERO_INV`| dst = 0 if src>thresh else src | Ters sıfırlama |
| `THRESH_OTSU` | Otomatik eşik hesaplama | Genel amaç |

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Otsu Eşikleme — 1979'nin Dönüm Noktası:</b><br><br>Nobuyuki Otsu, 1979'da "A Threshold Selection Method from Gray-Level Histograms" makalesini yayımladı. Fikir: Histogram iki bölgeye (ön plan + arka plan) ayrıldığında sınıflar arası varyansı maksimize eden eşiği bul.<br><br>Bu makale bilgisayarlı görüde <b>en çok atıf alan çalışmalardan</b> biri oldu. 40+ yıl sonra hâlâ standart referans olarak kullanılıyor.<br><br>Matematiksel özü: Histogram bimodal (iki tepeli) olduğunda mükemmel çalışır. Bu neden belge taramada iyidir? Beyaz kağıt + siyah mürekkep = iki belirgin tepe.<br><br>Sınırlama: Tek bir global eşik kullanır. Görüntü boyunca aydınlatma eşit değilse (eğik ışık, gölge) başarısız olur — bunun çözümü adaptif eşiklemedir!</span></div>

In [ ]:
import cv2 as cv

resim = cv.imread("resim/sudoku.jpg", cv.IMREAD_GRAYSCALE)

# Basit ikili eşikleme
ret, esiklenmis = cv.threshold(resim, 50, 255, cv.THRESH_BINARY)
print(f"Eşik değeri: {ret}")

cv.imshow("esiklenmis", esiklenmis)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
import cv2 as cv

resim = cv.imread("resim/sudoku.jpg", cv.IMREAD_GRAYSCALE)

# Ters ikili: siyah ve beyaz yer değişir
ret, esiklenmis = cv.threshold(resim, 50, 255, cv.THRESH_BINARY_INV)

cv.imshow("esiklenmis", esiklenmis)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
import cv2 as cv

resim = cv.imread("resim/sudoku.jpg", cv.IMREAD_GRAYSCALE)

# TRUNC: eşiğin üstündeki değerleri eşiğe sabitle
ret, esiklenmis = cv.threshold(resim, 50, 255, cv.THRESH_TRUNC)

cv.imshow("esiklenmis", esiklenmis)
cv.waitKey(0)
cv.destroyAllWindows()

## 6.2 Tüm Eşikleme Yöntemlerini Karşılaştırma — matplotlib

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread("resim/sudoku.jpg", 0)

# Sabit eşik türleri
ret, th1 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
ret, th2 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY_INV)
ret, th3 = cv2.threshold(img, 127, 255, cv2.THRESH_TRUNC)
ret, th4 = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO)
ret, th5 = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO_INV)

titles = ["Orijinal", "BINARY", "BINARY_INV", "TRUNC", "TOZERO", "TOZERO_INV"]
images = [img, th1, th2, th3, th4, th5]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
fig.suptitle("Eşikleme Yöntemleri Karşılaştırması (eşik=127)", fontsize=13)
for ax, (title, image) in zip(axes.flat, zip(titles, images)):
    ax.imshow(image, "gray")
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

---
# 🧠 BÖLÜM 7: Adaptif Eşikleme

## 7.1 Global Eşiklemenin Sorunu

```
Global eşikleme:           Adaptif eşikleme:

Tüm görüntü için tek eşik   Her bölge için ayrı eşik
  ┌────────────────┐          ┌────┬────┬────┬────┐
  │                │          │127 │ 95 │140 │110 │
  │   eşik = 127   │          ├────┼────┼────┼────┤
  │                │          │ 88 │120 │ 75 │130 │
  └────────────────┘          └────┴────┴────┴────┘

Gölge veya eşit olmayan    Her bölge kendi aydınlatmasına
aydınlatmada başarısız     göre optimize edilir
```

`cv.adaptiveThreshold(src, maxValue, adaptiveMethod, thresholdType, blockSize, C)`

| Parametre | Açıklama |
|-----------|----------|
| `blockSize` | Yerel alanın boyutu (tek sayı, ≥3) |
| `C` | Ortalamadan çıkarılacak sabit (ince ayar) |
| `ADAPTIVE_THRESH_MEAN_C` | Komşuların ortalaması - C |
| `ADAPTIVE_THRESH_GAUSSIAN_C` | Gaussian ağırlıklı ortalama - C |

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread("resim/sudoku.jpg", 0)

# Farklı parametrelerle adaptif eşikleme
th1 = cv2.adaptiveThreshold(img, 255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY, 11, 4)

th2 = cv2.adaptiveThreshold(img, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 17, 6)

th3 = cv2.adaptiveThreshold(img, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 17, 9)

th4 = cv2.adaptiveThreshold(img, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 17, 18)

# Otsu: Gaussian blur + otomatik eşik
blur = cv2.GaussianBlur(img, (3, 3), 0)
ret3, th5 = cv2.threshold(blur, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f"Otsu otomatik eşik değeri: {ret3}")

titles = ["Orijinal",
          "MEAN C=4",
          "GAUSSIAN C=6",
          "GAUSSIAN C=9",
          "GAUSSIAN C=18",
          f"OTSU (otomatik={ret3:.0f})"]
images = [img, th1, th2, th3, th4, th5]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
fig.suptitle("Adaptif Eşikleme Yöntemleri Karşılaştırması", fontsize=13)
for ax, (title, image) in zip(axes.flat, zip(titles, images)):
    ax.imshow(image, "gray")
    ax.set_title(title, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>blockSize ve C Parametrelerinin Geometrik Anlamı:</b><br><br><b>blockSize=11:</b> Her piksel için 11×11 = 121 komşu piksel incelenir. Küçük blockSize (3,5) → ince detayları yakalar ama gürültüye duyarlı. Büyük blockSize (51,101) → geniş ışık gradyanını dengeler ama ince çizgileri kaybeder.<br><br><b>C sabiti:</b> Hesaplanan yerel eşikten C kadar çıkarılır. Pozitif C → eşik düşer → daha fazla piksel "ön plan" → görüntü daha açık. C=0 → salt ortalama. C=5-10 → tipik değer.<br><br><b>Gerçek dünya kural of thumb:</b><br>• Metin/belge tarama → blockSize=11-25, C=5-10<br>• El yazısı → blockSize=31-51, C=15-20<br>• Tıbbi görüntü → Otsu veya deney ile belirle<br>• Parmak izi → blockSize=7-11, C=2-5</span></div>

In [ ]:
import cv2 as cv

# Adaptif eşikleme ile fare takibi
video = cv.VideoCapture("video/rat.avi")

while video.isOpened():
    ret, frame = video.read()
    if not ret:
        break

    grayFrame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    grayFrame  = grayFrame[20:570, 230:770]   # ROI

    # Adaptif Gaussian eşikleme
    esiklenmis = cv.adaptiveThreshold(
        grayFrame, 255,
        cv.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv.THRESH_BINARY,
        blockSize=25,
        C=25
    )

    cv.imshow("fare takip (adaptif)", esiklenmis)
    if cv.waitKey(33) == 27:
        break

video.release()
cv.destroyAllWindows()

---
# 🐀 BÖLÜM 8: Fare Takibi — Eşikleme ile

## 8.1 Fare Yolu Analizi — Temel Versiyon

Statik eşikleme ile beyaz fareyi siyah arenada takip etme:

In [ ]:
import cv2 as cv

video = cv.VideoCapture("video/rat.avi")

while video.isOpened():
    ret, frame = video.read()
    if not ret:
        break

    # Gri + ROI
    grayFrame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    grayFrame = grayFrame[20:570, 230:770]

    # Statik eşik: 180 üstü beyaz (fare)
    ret2, esiklenmis = cv.threshold(grayFrame, 180, 255,
                                    cv.THRESH_BINARY)

    cv.imshow("fare takip", esiklenmis)
    if cv.waitKey(33) == 27:
        quit()

video.release()
cv.destroyAllWindows()

## 8.2 Öğrenci Çözümleri — Farklı Yaklaşımlar

Aynı problemi farklı yöntemlerle çözen öğrenci kodları. Her birinin artı ve eksilerini inceleyelim:

In [ ]:
# ── Yaklaşım A: HSV maskeleme ile ──────────────────────────────────────
"""
Farenin siyah olmayan piksellerini HSV uzayında bul.
"""
import cv2 as cv
import numpy as np

video = cv.VideoCapture("video/rat.avi")

lower_white = np.array([0, 0, 0])
upper_white = np.array([255, 10, 255])   # düşük doygunluk = gri/beyaz

while video.isOpened():
    ret, frame = video.read()
    if not ret:
        break

    siyah = frame[20:570, 220:765]
    hsv   = cv.cvtColor(siyah, cv.COLOR_BGR2HSV)
    mask  = cv.inRange(hsv, lower_white, upper_white)
    res   = cv.bitwise_and(siyah, siyah, mask=mask)

    ret2, esiklenmis = cv.threshold(res, 30, 255, cv.THRESH_BINARY_INV)

    cv.imshow("frame", frame)
    cv.imshow("mask",  mask)
    cv.imshow("res",   esiklenmis)

    if cv.waitKey(1) == ord("q"):
        break

video.release()
cv.destroyAllWindows()

In [ ]:
# ── Yaklaşım B: Doğrudan eşik — Sade ama etkili ──────────────────────
import numpy as np
import cv2 as cv

video = cv.VideoCapture("video/rat.avi")

lower_white = np.array([0, 0, 0])
upper_white = np.array([255, 255, 100])

while True:
    ret, frame = video.read()
    if not ret:
        break

    frame2 = frame[:, 200:790]
    hsv    = cv.cvtColor(frame2, cv.COLOR_BGR2HSV)
    mask   = cv.inRange(hsv, lower_white, upper_white)

    cv.imshow("aaaaaaa", mask)
    if cv.waitKey(1) == ord("q"):
        break

video.release()
cv.destroyAllWindows()

In [ ]:
# ── Yaklaşım C: Kameradan kırmızıyı beyaza dönüştür ───────────────────
import cv2 as cv
import numpy as np

video = cv.VideoCapture(0)

lower_red = np.array([160, 100, 100])
upper_red = np.array([179, 255, 255])

while video.isOpened():
    ret, frame = video.read()
    if not ret:
        break

    hsv  = cv.cvtColor(frame, cv.COLOR_BGR2HSV)
    mask = cv.inRange(hsv, lower_red, upper_red)
    # mask: kırmızı=255(beyaz), diğer=0(siyah)

    cv.imshow("kamera", mask)
    if cv.waitKey(33) == ord("q"):
        break

video.release()
cv.destroyAllWindows()

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Hangi yaklaşım ne zaman?</b><br><br><b>Statik eşik (THRESH_BINARY):</b> Aydınlatma kontrollüyse ve nesne-arka plan kontrastı yüksekse en hızlı çözüm. Fare takibinde beyaz fare + siyah arena kombinasyonu bu yöntemi ideal kılar.<br><br><b>HSV maskeleme (inRange):</b> Nesnenin rengi biliniyor ve aydınlatma değişkenlik gösteriyorsa daha sağlam. Dış ortam uygulamalarında tercih edilir.<br><br><b>Adaptif eşik:</b> Aydınlatma eşit değilse veya nesne yoğunluğu bilinmiyorsa. Daha yavaş ama en genel çözüm.<br><br><b>MOG2/KNN (Arka Plan Çıkarma):</b> Nesne sabit, arka plan hareketliyse veya uzun süreli takipte. OpenCV'nin en güçlü takip aracı.</span></div>

---
# ✨ BÖLÜM 9: Şeffaf Renk Seçimi — Gelişmiş Uygulama

## 9.1 Tıklanan Pikselin Rengini Şeffaf Yapma

Kameradan görüntü al, BOŞLUK ile dondur, fare ile tıklanan rengi şeffaf yap:

In [ ]:
"""
Kameradan canlı görüntü alır.
BOŞLUK → anlık kareyi dondurur, fare ile tıklanan rengi şeffaf yapar.
R       → canlı yayına geri döner.
Q / ESC → çıkış.
"""

import cv2
import numpy as np

TOLERANS = 30   # renk yakınlık eşiği (0-255)


def rengi_seffaf_yap(goruntu_bgr, b, g, r, tolerans):
    """Verilen BGR rengine yakın pikselleri şeffaf yapar, BGRA döndürür."""
    bgra = cv2.cvtColor(goruntu_bgr, cv2.COLOR_BGR2BGRA)

    alt = np.array([max(0, b-tolerans), max(0, g-tolerans),
                    max(0, r-tolerans), 0], dtype=np.uint8)
    ust = np.array([min(255, b+tolerans), min(255, g+tolerans),
                    min(255, r+tolerans), 255], dtype=np.uint8)

    maske = cv2.inRange(bgra, alt, ust)
    bgra[maske > 0, 3] = 0   # alpha=0: şeffaf

    etkilenen = int(np.count_nonzero(maske))
    toplam    = goruntu_bgr.shape[0] * goruntu_bgr.shape[1]
    print(f"Şeffaf piksel: {etkilenen}/{toplam} "
          f"(%{etkilenen/toplam*100:.1f})")
    return bgra


def onizleme_olustur(bgra):
    """Şeffaf alanları damalı desen üzerine harmanlayarak BGR önizleme üretir."""
    y, x = bgra.shape[:2]
    d = 10
    zemin = np.zeros((y, x, 3), dtype=np.uint8)
    for i in range(0, y, d):
        for j in range(0, x, d):
            renk = 200 if (i // d + j // d) % 2 == 0 else 255
            zemin[i:i+d, j:j+d] = renk

    alfa    = bgra[:, :, 3:4].astype(float) / 255.0
    bgr_f   = bgra[:, :, :3].astype(float)
    zemin_f = zemin.astype(float)
    return (bgr_f * alfa + zemin_f * (1 - alfa)).astype(np.uint8)


donuk_kare = None
mod        = "canli"


def fare_callback(event, x, y, flags, param):
    global donuk_kare
    if event != cv2.EVENT_LBUTTONDOWN or donuk_kare is None:
        return
    b, g, r = [int(v) for v in donuk_kare[y, x]]
    print(f"Tıklanan ({x},{y}): BGR=({b},{g},{r})")
    sonuc    = rengi_seffaf_yap(donuk_kare, b, g, r, TOLERANS)
    onizleme = onizleme_olustur(sonuc)
    cv2.imshow("Donuk Pencere", onizleme)
    cv2.imwrite("seffaf_cikti.png", sonuc)
    print("Kaydedildi: seffaf_cikti.png")


kamera = cv2.VideoCapture(0)
cv2.namedWindow("Canli Kamera", cv2.WINDOW_NORMAL)

print("BOŞLUK=Dondur | Sol tık=Renk seç | R=Geri dön | Q=Çıkış")

while True:
    if mod == "canli":
        ret, kare = kamera.read()
        if not ret:
            break
        cv2.imshow("Canli Kamera", kare)

    tus = cv2.waitKey(1) & 0xFF

    if tus in (ord("q"), ord("Q"), 27):
        break
    elif tus == ord(" ") and mod == "canli":
        ret, donuk_kare = kamera.read()
        if ret:
            mod = "donuk"
            cv2.namedWindow("Donuk Pencere", cv2.WINDOW_NORMAL)
            cv2.setMouseCallback("Donuk Pencere", fare_callback)
            cv2.imshow("Donuk Pencere", donuk_kare)
            print("Kare donduruldu — renk seçmek için tıklayın")
    elif tus in (ord("r"), ord("R")) and mod == "donuk":
        cv2.destroyWindow("Donuk Pencere")
        donuk_kare = None
        mod = "canli"
        print("Canlı yayına döndü")

kamera.release()
cv2.destroyAllWindows()

<div style="background: #2d0020; border-left: 5px solid #f72585; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f72585; font-size: 1.08em;">🌍 Gerçek Dünya</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Renk Tabanlı Şeffaflık — Yeşil Ekran (Chroma Key):</b><br><br>Bu uygulamanın endüstriyel ölçekteki kardeşi <b>chroma keying</b>dir. Film setlerindeki yeşil ekran (green screen / chroma key) tam bu mantıkla çalışır: belirli yeşil tonu piksellerini tespit et, şeffaf yap, yerine başka bir arka plan yerleştir.<br><br>Neden yeşil? İnsan derisi tonları yeşil kanalda en az bileşene sahiptir — oyuncu ile arka plan arasındaki renk farkı maksimum olur. Mavi de kullanılır (bluescreen), ama gece çekimleri ve lacivert kostümler sorun çıkarır.<br><br>Hollywood blokbaster filmlerden Zoom arka plan değiştiricisine, hava durumu sunucusundan oyun yayıncılarına kadar milyarlarca dolarlık endüstri bu basit fikre dayanır.</span></div>

---
# 🔀 BÖLÜM 10: Görüntü Bölgesi Kopyalama

## 10.1 ROI'yi Başka Görüntüye Kopyalama

Bir görüntünün belirli bölgesini başka bir görüntüye yerleştirme:

In [ ]:
import cv2 as cv

resim_sudoku = cv.imread("resim/sudoku.jpg")
resim_bugday = cv.imread("resim/bugday.jpg")

print("Sudoku:", resim_sudoku.shape)
print("Buğday:", resim_bugday.shape)

# Sudoku'dan 50:400, 50:200 bölgesini
# Buğday'ın aynı bölgesine kopyala
resim_bugday[50:400, 50:200] = resim_sudoku[50:400, 50:200]

cv.imshow("kirpilmis resim", resim_bugday)
cv.waitKey(0)
cv.destroyAllWindows()

print("\nÖnemli: Bu işlem resim_bugday'ı yerinde değiştirdi!")
print("Orijinali korumak için önce resim_bugday.copy() kullanın.")

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>ROI Kopyalamanın Yaygın Kullanımları:</b><br><br><b>Görüntü birleştirme (image compositing):</b> Farklı görüntülerden bölgeler alarak yeni bir kolaj oluşturma. Gazetecilikte "fotoğraf manipülasyonu" tespiti tam bu işlemin izini bulmaya çalışır.<br><br><b>Nesne ekleme (object insertion):</b> Kesilmiş nesneyi (örn. bir ürün) farklı bir arka plana yerleştirme — e-ticaret fotoğrafçılığı.<br><br><b>Marka/logo ekleme (watermarking):</b> Videoların belirli köşesine logo çerçeve çerçeve kopyalama.<br><br><b>Veri gizleme (redaction):</b> Hassas belgelerde kişisel bilgilerin üstüne siyah dikdörtgen kopyalama. Mahkeme belgelerindeki sansür işlemi tam bu teknikle yapılır.</span></div>

---
# 🏛️ BÖLÜM 11: Üniversite Seviyesi Alıştırmalar

Aşağıdaki sorular bu haftanın konularını ileri düzey bilgisayar görüsü ve mühendislik perspektifiyle birleştirir. **Cevaplar verilmemiştir.**

---
## ❓ Soru 1 — Renk Uzayı Karşılaştırmalı Nesne Takip Sistemi

**Zorluk: ⭐⭐⭐☆☆**

Aynı renkli nesneyi (örn. turuncu top) **dört farklı renk uzayında** aynı anda takip eden ve her uzayın başarı oranını ölçen karşılaştırmalı bir sistem yazın.

### Görev:
1. Kameradan kare oku
2. Her kareyi şu uzaylara çevir: BGR, HSV, LAB, YCrCb
3. Her uzayda `cv.inRange` ile renk maskesi oluştur
4. Maskedeki beyaz piksel sayısını "tespit skoru" olarak kaydet
5. Dört pencereyi ve tespit skorlarını aynı anda göster
6. Her 100 karede ortalama tespit skorunu yazdır ve "en başarılı uzay" belirle

### Düşünmeniz Gereken Sorular:
- Farklı aydınlatma koşullarında (güneş, floresan, karanlık) hangi uzay en kararlı?
- LAB uzayında nesne tespiti için hangi kanallar kullanılır? Neden?
- Renk tutarsızlığını azaltmak için histogram eşitleme ne zaman yardımcı olur?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

# Her uzay için aralıklar (deney yaparak belirleyin)
HSV_aralik   = ([10, 100, 100], [25, 255, 255])   # turuncu
BGR_aralik   = ([0, 50, 150],  [80, 150, 255])    # turuncu BGR
LAB_aralik   = ([20, 130, 150],[240, 170, 200])   # turuncu LAB
YCrCb_aralik = ([0, 150, 100], [255, 180, 140])   # turuncu YCrCb

video = cv.VideoCapture(0)
# TODO: döngü ve karşılaştırma
```

---
## ❓ Soru 2 — Sudoku Çözücü — Görüntü İşleme Ön Adımı

**Zorluk: ⭐⭐⭐⭐☆**

Bir Sudoku fotoğrafından ızgara hücrelerini otomatik olarak çıkaran görüntü işleme pipeline'ı tasarlayın. (Sudoku çözme algoritması yazmanıza gerek yok — yalnızca görüntü ön işleme)

### Görev:
1. Gri + Gaussian blur + adaptif eşikleme
2. `cv.findContours()` ile ızgara hücrelerini bul
3. Alan filtresi: çok küçük veya çok büyük konturları at
4. Her hücre için bounding box al, `cv.resize(56, 56)` ile normalize et
5. 81 hücrenin görüntüsünü `9×9` montaj görüntüsü olarak göster
6. Her hücrenin içindeki rakamı `cv.Canny + cv.HoughLinesP` ile tespit et (boş hücre = çizgi yok)

### Düşünmeniz Gereken Sorular:
- Perspektif bozukluğu olan Sudoku fotoğrafında ızgara tespiti neden zorlaşır?
- Konturu bounding box'a dönüştürmek her zaman hücre sınırını doğru verir mi? Kareye yakın olmayan konturlar nasıl elenir?
- Bu pipeline ile OCR (optik karakter tanıma) entegre edilerek tam çözüm nasıl tamamlanır?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def sudoku_hucrelerini_cikar(goruntu_yolu):
    img = cv.imread(goruntu_yolu, cv.IMREAD_GRAYSCALE)
    # 1. Ön işleme
    # 2. Kontur tespiti
    # 3. Hücre kırpma
    # 4. Montaj görüntüsü
    pass
```

---
## ❓ Soru 3 — Gerçek Zamanlı Yeşil Ekran (Chroma Key) Uygulaması

**Zorluk: ⭐⭐⭐⭐☆**

Kameradan gelen görüntüdeki belirli rengi (yeşil/mavi ekran) başka bir arka plan görüntüsüyle gerçek zamanlı olarak değiştirin.

### Görev:
1. HSV uzayında yeşil maskeyi oluştur
2. Maskeyi yumuşat: `cv.GaussianBlur` + `cv.morphologyEx(MORPH_CLOSE)` ile kenarları temizle
3. Alpha blending formülü:
   `sonuç = ön_plan × (1 - alpha) + arka_plan × alpha`
   Burada `alpha` maskenin 0-1 arasındaki değeri
4. Arka planı her karede döndür: `cv.VideoCapture("arkaplan.mp4")`
5. `h` tuşu: HSV aralığını trackbar ile ayarla
6. Saçak (fringe) efektini gidermek için kenar piksellerine ek blur uygula

### Düşünmeniz Gereken Sorular:
- "Kenar saçakları" (green spill) neden oluşur ve nasıl giderilir?
- Donanım destekli chroma key (broadcast mixer) yazılımdan farkı ne?
- Zoom'un sanal arka plan özelliği yeşil ekransız nasıl çalışır? (İpucu: derin öğrenme tabanlı insan segmentasyonu)

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

# Yeşil ekran aralığı
lower_green = np.array([40, 70, 70])
upper_green = np.array([80, 255, 255])

def chroma_key(on_plan, arka_plan, lower, upper):
    """on_plan'daki yeşili arka_plan ile değiştir."""
    # TODO
    pass

video_on    = cv.VideoCapture(0)
video_arka  = cv.VideoCapture("arkaplan.mp4")
# TODO: döngü
```

---
## ❓ Soru 4 — İnsan Takibi: Renk Histogramı ile Re-Identification

**Zorluk: ⭐⭐⭐⭐⭐**

Kamera görüntüsünde kullanıcının fareyle seçtiği kişiyi renk histogramı benzerliğiyle sürekli takip eden bir sistem yazın. Kişi geçici olarak görüş alanından çıkıp tekrar girince yeniden tanımlayabilmeli.

### Görev:
1. Fare ile ilk kareye tıklandığında 50×50 piksellik "hedef bölge" seç
2. Hedef bölgenin HSV histogramını hesapla ve kaydet (referans histogramı)
3. Her yeni karede kayan pencere ile görüntüyü tara (stride=10px)
4. Her pencere bölgesi için HSV histogramı hesapla, `cv.compareHist(HISTCMP_BHATTACHARYYA)` ile referansla karşılaştır
5. En düşük mesafeli (en benzer) bölgeyi hedef olarak işaretle ve takip et
6. Tarih penceresi (temporal smoothing): son 5 konumun ortalaması alınarak "atlayan" takibi stabilize et
7. Kişi kaybolunca 3 saniye sonra "kaybedildi" uyarısı ver

### Düşünmeniz Gereken Sorular:
- Renk histogramı ile kişi takibi hangi durumlarda başarısız olur?
- CamShift algoritması bu sistemi nasıl daha verimli çözer?
- Modern nesne takip algoritmaları (SORT, DeepSORT) bu yaklaşımdan nasıl farklı? Derin öğrenme katkısı ne?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np
from collections import deque

hedef_hist   = None   # referans histogramı
konum_gecmis = deque(maxlen=5)   # son 5 konum

def histogram_hesapla(bolge):
    """HSV histogramı hesapla, normalize et."""
    hsv = cv.cvtColor(bolge, cv.COLOR_BGR2HSV)
    hist = cv.calcHist([hsv], [0, 1], None, [30, 32], [0,180,0,256])
    cv.normalize(hist, hist)
    return hist

video = cv.VideoCapture(0)
# TODO: döngü, kayan pencere, karşılaştırma
```

---
## ❓ Soru 5 — Özellik Tabanlı Görüntü Hizalama (Image Registration)

**Zorluk: ⭐⭐⭐⭐⭐**

Aynı sahnenin farklı açılardan çekilmiş iki fotoğrafını özellik eşleştirme yöntemiyle hizalayın (register edin). Bu, tıbbi görüntülemede, uydu görüntülerinde ve panorama birleştirmede temel adımdır.

### Görev:
1. İki görüntüyü gri tonlamaya çevir
2. `cv.ORB_create()` ile ORB özellik noktaları ve tanımlayıcılar hesapla
3. `cv.BFMatcher()` ile tanımlayıcıları eşleştir, Lowe's ratio test uygula
4. En az 10 iyi eşleşme varsa `cv.findHomography(RANSAC)` ile homografi matrisi hesapla
5. `cv.warpPerspective()` ile ikinci görüntüyü birinciye hizala
6. Hizalanmış görüntü ile orijinalin farkını görselleştir
7. Hizalama kalitesini `cv.matchTemplate` ile sayısal olarak ölç

### Düşünmeniz Gereken Sorular:
- ORB neden SIFT/SURF'e kıyasla tercih edilebilir? (Patent, hız, doğruluk)
- RANSAC algoritması yanlış eşleşmeleri (outlier) nasıl eliyor?
- Tıbbi MRI seansları arasındaki görüntü hizalamada rigid vs non-rigid registration farkı nedir?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def goruntu_hizala(img1_yolu, img2_yolu, max_ozellik=5000):
    """
    img2'yi img1'e hizalar.
    Döndürür: (hizalanmis_img2, eslesmeler, homografi_matrisi)
    """
    img1 = cv.imread(img1_yolu, cv.IMREAD_GRAYSCALE)
    img2 = cv.imread(img2_yolu, cv.IMREAD_GRAYSCALE)

    # ORB dedektör
    orb = cv.ORB_create(max_ozellik)
    # TODO: özellik çıkarma, eşleştirme, homografi, warpPerspective
    pass
```

---
<div style="background: linear-gradient(135deg, #050010 0%, #0a0025 40%, #002020 100%);padding: 38px 42px; border-radius: 18px; text-align: center; margin-top: 20px; border: 1px solid #00ffaa33;"><h2 style="color: #00ffaa; font-family: 'Courier New', monospace; margin: 0 0 22px 0;">🎯 7. Hafta Özeti</h2><div style="color: #cdd6f4; text-align: left; max-width: 700px; margin: 0 auto; line-height: 2.2;">✅ <b>Renk uzayları:</b> BGR, HSV, LAB, YCrCb — tarihçe ve matematiksel temel<br>✅ <b>BGRA / Alpha:</b> 4. kanal, straight vs premultiplied alpha farkı<br>✅ <b>Parlaklık bazlı şeffaflık:</b> piksel döngüsü → NumPy boolean indexing<br>✅ <b>HSV maskeleme:</b> inRange + bitwise_and — aydınlatmadan bağımsız renk tespiti<br>✅ <b>Kırmızı iki aralık:</b> Renk çemberi başı ve sonu — bitwise_or ile birleştirme<br>✅ <b>Perspektif dönüşüm:</b> getPerspectiveTransform + warpPerspective<br>✅ <b>Median Blur:</b> Gaussian'dan farkı — tuz-biber gürültüsü için üstün<br>✅ <b>Eşikleme (5 tip):</b> BINARY, BINARY_INV, TRUNC, TOZERO, TOZERO_INV<br>✅ <b>Otsu eşikleme:</b> 1979 makalesi, histogram bimodal varsayımı, sınırlar<br>✅ <b>Adaptif eşikleme:</b> blockSize ve C parametreleri, MEAN vs GAUSSIAN<br>✅ <b>Chroma key:</b> Yeşil ekranın sinema tarihindeki yeri ve HSV bağlantısı<br>✅ <b>ROI kopyalama:</b> Görüntü bölgesi yerleştirme — kompoziting temeli</div><p style="color: #6b7280; margin: 22px 0 0 0; font-family: 'Courier New', monospace;">8. Hafta → Geometrik Dönüşümler: Resize, Flip, Rotate & Affine 📐</p></div>